# FGSM Phase 2 OCRA
Converted from `FGSM_Phase2_OCRA.py`. Run the code cell below in the `vlm_ftune` conda environment.

In [2]:
import json
import os
import re
import time
from collections import Counter
from difflib import SequenceMatcher
from io import BytesIO

import cv2
import numpy as np
import torch
from PIL import Image, ImageFilter
from skimage.restoration import denoise_tv_chambolle
from transformers import AutoModelForCausalLM, AutoProcessor


# ============================================================
# Configuration
# ============================================================
IMAGE_DIR = "dataset/val2017"
NUM_IMAGES = 500
EPS = 0.03

MODEL_NAME = "microsoft/Florence-2-base"
MODEL_REVISION = "refs/pr/26"
EXPECTED_CONDA_ENV = "vlm_ftune"

LOG_PATH = "FGSM_Phase2_OCRA.log"
JSON_PATH = "FGSM_Phase2_OCRA.json"

JPEG_QUALITY = 75
GAUSSIAN_SIGMA = 1.0
TVM_WEIGHT = 0.05
MEDIAN_KERNEL = 3


def setup_device():
    if torch.cuda.is_available():
        return torch.device("cuda:0"), torch.float16
    return torch.device("cpu"), torch.float32


device, torch_dtype = setup_device()


def assert_expected_environment():
    """Ensure script runs in the validated conda env to avoid venv dependency mismatches."""
    current = os.environ.get("CONDA_DEFAULT_ENV", "")
    if current != EXPECTED_CONDA_ENV:
        raise RuntimeError(
            f"This script must run in conda env '{EXPECTED_CONDA_ENV}'. "
            f"Current env: '{current or 'unknown'}'. "
            "Run with: conda run -n vlm_ftune python FGSM_Phase2_OCRA.py"
        )


def log(msg, fh):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line)
    fh.write(line + "\n")
    fh.flush()


def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def text_similarity(a, b):
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()


def decode_ocr(parsed_or_text):
    if isinstance(parsed_or_text, dict):
        val = parsed_or_text.get("<OCR>", "")
        if isinstance(val, str):
            return val
        if isinstance(val, list):
            return " ".join([str(x) for x in val])
        if isinstance(val, dict):
            if "text" in val:
                return str(val["text"])
            return json.dumps(val, ensure_ascii=True)
    return str(parsed_or_text)


def sanitize_generated_text(raw):
    cleaned = re.sub(r"<[^>]+>", " ", raw)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def pil_to_arr(pil_img):
    return np.array(pil_img)


def arr_to_pil(arr):
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


def defend_jpeg(pil_img):
    buf = BytesIO()
    pil_img.save(buf, format="JPEG", quality=JPEG_QUALITY)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def defend_median(pil_img):
    return pil_img.filter(ImageFilter.MedianFilter(size=MEDIAN_KERNEL))


def defend_gaussian(pil_img):
    return pil_img.filter(ImageFilter.GaussianBlur(radius=GAUSSIAN_SIGMA))


def defend_tvm(pil_img):
    arr = pil_to_arr(pil_img).astype(np.float64) / 255.0
    denoised = denoise_tv_chambolle(arr, weight=TVM_WEIGHT, channel_axis=-1)
    return arr_to_pil(denoised * 255.0)


def defend_blur_tvm(pil_img):
    return defend_tvm(defend_gaussian(pil_img))


def vote_text(texts):
    norm = [normalize_text(t) for t in texts]
    counts = Counter(norm)
    best_count = max(counts.values())
    candidates = [k for k, v in counts.items() if v == best_count]
    if len(candidates) == 1:
        target = candidates[0]
    else:
        target = sorted(candidates, key=lambda x: len(x), reverse=True)[0]
    for t in texts:
        if normalize_text(t) == target:
            return t
    return texts[0]


def get_ocr_text(model, processor, pil_img):
    img_w, img_h = pil_img.size
    with torch.no_grad():
        inputs = processor(text="<OCR>", images=pil_img, return_tensors="pt")
        input_ids = inputs.input_ids.to(device)
        pixel_values = inputs.pixel_values.to(device=device, dtype=torch_dtype)
        gen_ids = model.generate(
            input_ids=input_ids,
            pixel_values=pixel_values,
            max_new_tokens=256,
            num_beams=3,
            do_sample=False,
        )
        raw = processor.batch_decode(gen_ids, skip_special_tokens=False)[0]
        try:
            parsed = processor.post_process_generation(raw, task="<OCR>", image_size=(img_w, img_h))
            out = decode_ocr(parsed)
        except Exception:
            out = sanitize_generated_text(raw)
    return out


def fgsm_attack_ocr(model, processor, pil_img, eps, img_mean, img_std):
    orig_size = pil_img.size
    inputs = processor(text="<OCR>", images=pil_img, return_tensors="pt")
    input_ids = inputs.input_ids.to(device)
    pixel_values = inputs.pixel_values.to(device=device, dtype=torch_dtype)

    with torch.no_grad():
        target_ids = model.generate(
            input_ids=input_ids,
            pixel_values=pixel_values,
            max_new_tokens=256,
            num_beams=3,
            do_sample=False,
        )
    target_ids = target_ids[:1, :].contiguous()

    pixel_values_adv = pixel_values.clone().detach().requires_grad_(True)
    outputs = model(input_ids=input_ids, pixel_values=pixel_values_adv, labels=target_ids)
    outputs.loss.backward()

    grad_sign = pixel_values_adv.grad.sign()
    adv_pixel_values = pixel_values.detach() + eps * grad_sign
    adv_pixel_values = torch.clamp(adv_pixel_values, -2.5, 2.5)

    mean = img_mean.squeeze(0)
    std = img_std.squeeze(0)
    adv_denorm = adv_pixel_values.squeeze(0) * std + mean
    adv_denorm = torch.clamp(adv_denorm, 0.0, 1.0)
    adv_np = (adv_denorm.permute(1, 2, 0).detach().cpu().float().numpy() * 255).astype(np.uint8)
    adv_pil = Image.fromarray(adv_np)
    if adv_pil.size != orig_size:
        adv_pil = adv_pil.resize(orig_size, Image.BICUBIC)
    return adv_pil


def run():
    assert_expected_environment()
    with open(LOG_PATH, "w", encoding="utf-8") as log_f:
        log("Starting FGSM OCR recovery run", log_f)
        log(f"Conda env: {os.environ.get('CONDA_DEFAULT_ENV', 'unknown')}", log_f)
        log(f"Device: {device}, dtype: {torch_dtype}", log_f)
        log(f"Image dir: {IMAGE_DIR}", log_f)
        log(f"Images: {NUM_IMAGES}, eps: {EPS}", log_f)

        files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        files = files[:NUM_IMAGES]
        if not files:
            raise RuntimeError(f"No images found in {IMAGE_DIR}")

        log(f"Loading Florence model: {MODEL_NAME}", log_f)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            revision=MODEL_REVISION,
            torch_dtype=torch_dtype,
            trust_remote_code=True,
        ).to(device)
        processor = AutoProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, trust_remote_code=True)

        img_mean = torch.tensor(processor.image_processor.image_mean, device=device, dtype=torch_dtype).view(1, 3, 1, 1)
        img_std = torch.tensor(processor.image_processor.image_std, device=device, dtype=torch_dtype).view(1, 3, 1, 1)

        ensembles = {
            "ens_blur_tvm_combo": ["jpeg", "blur_tvm", "median"],
            "ens_jpeg_median_gaussian": ["jpeg", "median", "gaussian"],
            "ens_jpeg_median_tvm": ["jpeg", "median", "tvm"],
        }
        branches = {
            "jpeg": defend_jpeg,
            "median": defend_median,
            "gaussian": defend_gaussian,
            "tvm": defend_tvm,
            "blur_tvm": defend_blur_tvm,
        }

        stats = {
            "images": len(files),
            "eps": EPS,
            "attacked_sim_sum": 0.0,
            "defenses": {name: {"sim_sum": 0.0, "recovery_sum": 0.0} for name in ensembles},
        }

        for i, fname in enumerate(files, 1):
            path = os.path.join(IMAGE_DIR, fname)
            clean_img = Image.open(path).convert("RGB")

            clean_text = get_ocr_text(model, processor, clean_img)
            adv_img = fgsm_attack_ocr(model, processor, clean_img, EPS, img_mean, img_std)
            attacked_text = get_ocr_text(model, processor, adv_img)

            attacked_sim = text_similarity(clean_text, attacked_text)
            stats["attacked_sim_sum"] += attacked_sim

            for ens_name, member_names in ensembles.items():
                member_texts = []
                for m in member_names:
                    img_m = branches[m](adv_img)
                    txt_m = get_ocr_text(model, processor, img_m)
                    member_texts.append(txt_m)

                defended_text = vote_text(member_texts)
                defended_sim = text_similarity(clean_text, defended_text)
                recovery = defended_sim - attacked_sim

                stats["defenses"][ens_name]["sim_sum"] += defended_sim
                stats["defenses"][ens_name]["recovery_sum"] += recovery

            if i % 10 == 0 or i == len(files):
                atk_avg = stats["attacked_sim_sum"] / i
                log(f"Progress {i}/{len(files)} | attacked_sim_avg={atk_avg:.4f}", log_f)

        n = len(files)
        attacked_avg = stats["attacked_sim_sum"] / n
        rows = []
        for name, v in stats["defenses"].items():
            sim_avg = v["sim_sum"] / n
            rec_avg = v["recovery_sum"] / n
            rows.append((name, sim_avg, rec_avg))
        rows.sort(key=lambda x: x[2], reverse=True)

        log("==== Final OCR Recovery Summary ====", log_f)
        log(f"Attacked OCR similarity avg: {attacked_avg:.4f}", log_f)
        for rank, (name, sim_avg, rec_avg) in enumerate(rows, 1):
            log(f"{rank}. {name}: defended_sim_avg={sim_avg:.4f}, recovery_avg={rec_avg:+.4f}", log_f)

        top3_inline = " | ".join([f"{r[0]}:sim={r[1]:.4f},rec={r[2]:+.4f}" for r in rows[:3]])
        log(f"TOP3 {top3_inline}", log_f)

        out = {
            "images": n,
            "eps": EPS,
            "attacked_similarity_avg": attacked_avg,
            "ranked_defenses": [
                {"rank": i + 1, "name": r[0], "defended_similarity_avg": r[1], "recovery_avg": r[2]}
                for i, r in enumerate(rows)
            ],
        }
        with open(JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(out, f, indent=2)

        log(f"Saved summary JSON: {JSON_PATH}", log_f)
        log(f"Saved concise log: {LOG_PATH}", log_f)


if __name__ == "__main__":
    run()

[01:54:41] Starting FGSM OCR recovery run
[01:54:41] Conda env: vlm_ftune
[01:54:41] Device: cuda:0, dtype: torch.float16
[01:54:41] Image dir: dataset/val2017
[01:54:41] Images: 500, eps: 0.03
[01:54:41] Loading Florence model: microsoft/Florence-2-base
[01:55:46] Progress 10/500 | attacked_sim_avg=0.4554
[01:56:00] Progress 20/500 | attacked_sim_avg=0.4044
[01:56:14] Progress 30/500 | attacked_sim_avg=0.3497
[01:56:27] Progress 40/500 | attacked_sim_avg=0.3940
[01:56:40] Progress 50/500 | attacked_sim_avg=0.3949
[01:56:52] Progress 60/500 | attacked_sim_avg=0.3746
[01:57:05] Progress 70/500 | attacked_sim_avg=0.3802
[01:57:18] Progress 80/500 | attacked_sim_avg=0.3669
[01:57:38] Progress 90/500 | attacked_sim_avg=0.3560
[01:57:52] Progress 100/500 | attacked_sim_avg=0.3443
[01:58:05] Progress 110/500 | attacked_sim_avg=0.3249
[01:58:20] Progress 120/500 | attacked_sim_avg=0.3267
[01:58:34] Progress 130/500 | attacked_sim_avg=0.3265
[01:58:46] Progress 140/500 | attacked_sim_avg=0.324